# Airline Operations & Disruption Intelligence
## Weather Data Collection & Cleaning

Collect historical hourly weather data for airports used in the cleaned BTS flight dataset.

**Period:** January 1, 2026 to March 31, 2026

**Weather variables:** temperature, precipitation, wind speed, weather code

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import requests
import json
import time
from pathlib import Path

## 2. Define project paths

In [2]:
project_path = Path(r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence")

flights_path = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\processed\flights_2026_q1.csv"
airport_path = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\airports\airport_reference.csv"

weather_raw_path = project_path / "data" / "raw" / "weather"
weather_processed_path = project_path / "data" / "processed"

weather_raw_path.mkdir(parents=True, exist_ok=True)
weather_processed_path.mkdir(parents=True, exist_ok=True)

print("Project path:", project_path)

Project path: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence


## 3. Load cleaned flight data

In [3]:
flights = pd.read_csv(flights_path)
print('Rows:', len(flights))
print('Columns:', len(flights.columns))
flights.head()

C:\Users\bhupa\AppData\Local\Temp\ipykernel_18772\1723287432.py:1: DtypeWarning: Columns (60,66) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv(flights_path)


Rows: 1847242
Columns: 70


,year,quarter,month,day_of_month,day_of_week,fl_date,mkt_unique_carrier,branded_code_share,mkt_carrier_airline_id,mkt_carrier,...,origin_timezone,dest_airport_name,dest_city,dest_country,dest_latitude,dest_longitude,dest_timezone,dep_hour,dep_minute,scheduled_departure
0,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,-5,Los Angeles International Airport,Los Angeles,United States,33.942501,-118.407997,-8,7,0,2026-01-01 07:00:00
1,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,-8,John F Kennedy International Airport,New York,United States,40.639801,-73.778900,-5,21,30,2026-01-01 21:30:00
2,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,-5,Louis Armstrong New Orleans International Airport,New Orleans,United States,29.993401,-90.258003,-6,22,45,2026-01-01 22:45:00
3,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,-7,Miami International Airport,Miami,United States,25.793200,-80.290604,-5,23,38,2026-01-01 23:38:00
4,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,-5,Charlotte Douglas International Airport,Charlotte,United States,35.214001,-80.943100,-5,7,57,2026-01-01 07:57:00


## 4. Check columns needed for weather integration

In [4]:
required_columns = ['origin', 'fl_date', 'crs_dep_time']
for column in required_columns:
    if column in flights.columns:
        print(column, '✓')
    else:
        print(column, 'MISSING')

origin ✓
fl_date ✓
crs_dep_time ✓


## 5. Load airport reference data

In [5]:
airport_ref = pd.read_csv(airport_path)
print('Rows:', len(airport_ref))
print('Columns:', len(airport_ref.columns))
airport_ref.head()

Rows: 362
Columns: 7


,iata_code,airport_name,city,country,latitude,longitude,timezone
0,PPG,Pago Pago International Airport,Pago Pago,American Samoa,-14.331000,-170.710007,-11
1,SPN,Saipan International Airport,Saipan,Northern Mariana Islands,15.119000,145.729004,10
2,GUM,Antonio B. Won Pat International Airport,Agana,Guam,13.483400,144.796005,10
3,STT,Cyril E. King Airport,St. Thomas,Virgin Islands,18.337299,-64.973396,-4
4,STX,Henry E Rohlsen Airport,St. Croix Island,Virgin Islands,17.701900,-64.798599,-4


## 6. Keep only columns needed for weather

In [6]:
airport_weather_ref = airport_ref[['iata_code', 'latitude', 'longitude']].copy()
airport_weather_ref.head()

,iata_code,latitude,longitude
0,PPG,-14.331000,-170.710007
1,SPN,15.119000,145.729004
2,GUM,13.483400,144.796005
3,STT,18.337299,-64.973396
4,STX,17.701900,-64.798599


## 7. Validate airport reference

In [7]:
print('Total airport records:', len(airport_weather_ref))
print('Unique airport codes:', airport_weather_ref['iata_code'].nunique())
print('\nMissing values:')
print(airport_weather_ref.isna().sum())
print('\nDuplicate airport codes:', airport_weather_ref['iata_code'].duplicated().sum())

Total airport records: 362
Unique airport codes: 362

Missing values:
iata_code    0
latitude     0
longitude    0
dtype: int64

Duplicate airport codes: 0


## 8. Identify airports used by flights

In [8]:
flight_airports = flights['origin'].dropna().astype(str).str.strip().unique()
flight_airports_set = set(flight_airports)
print('Unique ORIGIN airports:', len(flight_airports))

Unique ORIGIN airports: 362


## 9. Check airport reference coverage

In [9]:
reference_airports_set = set(airport_weather_ref['iata_code'])
missing_airports = flight_airports_set - reference_airports_set
print('Airports missing from reference:', len(missing_airports))
print(missing_airports)

Airports missing from reference: 0
set()


Do not silently remove unmatched airports. Investigate and verify them first. XWA and EAR were previously verified and added to the reference table.

## 10. Create final airport list

In [10]:
weather_airports = airport_weather_ref[airport_weather_ref['iata_code'].isin(flight_airports_set)].copy()
print('Airports requiring weather data:', len(weather_airports))
weather_airports.head(20)

Airports requiring weather data: 362


,iata_code,latitude,longitude
0,PPG,-14.331000,-170.710007
1,SPN,15.119000,145.729004
2,GUM,13.483400,144.796005
3,STT,18.337299,-64.973396
4,STX,17.701900,-64.798599
5,BQN,18.494900,-67.129402
6,PSE,18.008301,-66.563004
7,SJU,18.439400,-66.001801
8,ITO,19.721399,-155.048004
9,FSM,35.336601,-94.367401


## 11. Validate coordinates

In [11]:
print('Missing latitude:', weather_airports['latitude'].isna().sum())
print('Missing longitude:', weather_airports['longitude'].isna().sum())
print('Duplicate airport codes:', weather_airports['iata_code'].duplicated().sum())

Missing latitude: 0
Missing longitude: 0
Duplicate airport codes: 0


## 12. Open-Meteo API settings

We make one request per airport for the full 90-day period, not one request per flight. The API response is requested in UTC; local-time handling will be addressed carefully during the later flight-weather join.

In [12]:
weather_url = 'https://archive-api.open-meteo.com/v1/archive'
start_date = '2026-01-01'
end_date = '2026-03-31'
weather_variables = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'weather_code']
print(weather_url, start_date, end_date)

https://archive-api.open-meteo.com/v1/archive 2026-01-01 2026-03-31


## 13. Test the API with ATL

In [13]:
atl = weather_airports[weather_airports['iata_code'] == 'ATL'].iloc[0]
atl_latitude = atl['latitude']
atl_longitude = atl['longitude']
print('Airport: ATL')
print('Latitude:', atl_latitude)
print('Longitude:', atl_longitude)

Airport: ATL
Latitude: 33.6367
Longitude: -84.428101


In [14]:
params = {
    'latitude': atl_latitude,
    'longitude': atl_longitude,
    'start_date': start_date,
    'end_date': end_date,
    'hourly': ','.join(weather_variables),
    'timezone': 'UTC'
}
response = requests.get(weather_url, params=params, timeout=60)
print('API Status Code:', response.status_code)

API Status Code: 200


In [15]:
if response.status_code == 200:
    print('API request successful.')
else:
    print('API request failed.')
    print(response.text)

API request successful.


In [16]:
data = response.json()
print(data.keys())
if 'hourly' in data:
    print(data['hourly'].keys())
    print(data['hourly']['time'][:5])
else:
    print(data)

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
dict_keys(['time', 'temperature_2m', 'precipitation', 'wind_speed_10m', 'weather_code'])
['2026-01-01T00:00', '2026-01-01T01:00', '2026-01-01T02:00', '2026-01-01T03:00', '2026-01-01T04:00']


## 14. Save the ATL raw response

In [17]:
atl_raw_file = weather_raw_path / 'ATL_2026-01-01_to_2026-03-31.json'
with open(atl_raw_file, 'w', encoding='utf-8') as file:
    json.dump(data, file, indent=4)
print('Raw API response saved:', atl_raw_file)

Raw API response saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\weather\ATL_2026-01-01_to_2026-03-31.json


## 15. Convert ATL response into a table

In [18]:
hourly = data['hourly']
atl_weather = pd.DataFrame({
    'time': hourly['time'],
    'temperature': hourly['temperature_2m'],
    'precipitation': hourly['precipitation'],
    'wind_speed': hourly['wind_speed_10m'],
    'weather_code': hourly['weather_code']
})
atl_weather['time'] = pd.to_datetime(atl_weather['time'])
atl_weather['date'] = atl_weather['time'].dt.date
atl_weather['hour'] = atl_weather['time'].dt.hour
atl_weather['airport_code'] = 'ATL'
atl_weather = atl_weather[['airport_code','date','hour','temperature','precipitation','wind_speed','weather_code']]
atl_weather.head()

,airport_code,date,hour,temperature,precipitation,wind_speed,weather_code
0,ATL,2026-01-01,0,6.7,0.0,12.1,0
1,ATL,2026-01-01,1,6.1,0.0,12.1,1
2,ATL,2026-01-01,2,5.5,0.0,13.0,0
3,ATL,2026-01-01,3,5.0,0.0,14.1,0
4,ATL,2026-01-01,4,4.4,0.0,14.1,0


## 16. Validate ATL test data

In [19]:
print('Total rows:', len(atl_weather))
print('Duplicate records:', atl_weather.duplicated(subset=['airport_code','date','hour']).sum())
print('\nMissing values:')
print(atl_weather.isna().sum())
print('\nNegative precipitation:', (atl_weather['precipitation'] < 0).sum())
print('Negative wind speed:', (atl_weather['wind_speed'] < 0).sum())

Total rows: 2160
Duplicate records: 0

Missing values:
airport_code     0
date             0
hour             0
temperature      0
precipitation    0
wind_speed       0
weather_code     0
dtype: int64

Negative precipitation: 0
Negative wind speed: 0


In [20]:
atl_weather_file = weather_processed_path / 'weather_ATL_2026_Q1.csv'
atl_weather.to_csv(atl_weather_file, index=False)
print('ATL test weather saved:', atl_weather_file)

ATL test weather saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\processed\weather_ATL_2026_Q1.csv


## 17. Collect weather for all airports

The code below uses an existing raw JSON file when available. This prevents repeated API requests if the notebook is restarted. New requests are followed by a short pause.

In [21]:
weather_results = []
failed_airports = []

In [22]:
for index, row in weather_airports.iterrows():
    airport_code = row['iata_code']
    latitude = row['latitude']
    longitude = row['longitude']

    print('\n' + '=' * 50)
    print(f'Processing {airport_code} ({index + 1}/{len(weather_airports)})')

    raw_file = weather_raw_path / f'{airport_code}_2026-01-01_to_2026-03-31.json'

    if raw_file.exists():
        print('Raw file already exists. Using saved file.')
        try: 
            with open(raw_file, 'r', encoding='utf-8') as file:
                data = json.load(file)
        except Exception as error:
            print('Could not read saved file:', error)
            failed_airports.append({'airport_code': airport_code, 'reason': 'Could not read saved JSON'})
            continue
    else:
        params = {
            'latitude': latitude,
            'longitude': longitude,
            'start_date': start_date,
            'end_date': end_date,
            'hourly': ','.join(weather_variables),
            'timezone': 'UTC'
        }
        try:
            response = requests.get(weather_url, params=params, timeout=60)
            print('API Status Code:', response.status_code)
        except requests.RequestException as error:
            print('API request failed:', error)
            failed_airports.append({'airport_code': airport_code, 'reason': str(error)})
            continue

        if response.status_code != 200:
            print('API failed.')
            failed_airports.append({'airport_code': airport_code, 'reason': response.text})
            continue

        data = response.json()
        if 'hourly' not in data:
            print('Hourly weather data missing.')
            failed_airports.append({'airport_code': airport_code, 'reason': 'hourly data missing'})
            continue

        with open(raw_file, 'w', encoding='utf-8') as file:
            json.dump(data, file, indent=4)
        print('Raw response saved.')
        time.sleep(1)

    hourly = data['hourly']
    weather_df = pd.DataFrame({
        'time': hourly['time'],
        'temperature': hourly['temperature_2m'],
        'precipitation': hourly['precipitation'],
        'wind_speed': hourly['wind_speed_10m'],
        'weather_code': hourly['weather_code']
    })
    weather_df['time'] = pd.to_datetime(weather_df['time'])
    weather_df['date'] = weather_df['time'].dt.date
    weather_df['hour'] = weather_df['time'].dt.hour
    weather_df['airport_code'] = airport_code
    weather_df = weather_df[['airport_code','date','hour','temperature','precipitation','wind_speed','weather_code']]
    weather_results.append(weather_df)
    print('Rows collected:', len(weather_df))


Processing PPG (1/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing SPN (2/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing GUM (3/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing STT (4/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing STX (5/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing BQN (6/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing PSE (7/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing SJU (8/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing ITO (9/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing FSM (10/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing GFK (11/362)
Raw file already exists. Using saved file.
Rows collected: 2160

Processing PRC (12/362)
Raw f

## 18. Combine all airports into ONE DataFrame

In [23]:
weather = pd.concat(weather_results, ignore_index=True)
print('Total weather rows:', len(weather))

Total weather rows: 781920


## 19. Validate the combined weather data

In [24]:
print('Shape:', weather.shape)
print('\nFirst 5 rows:')
display(weather.head())
print('\nLast 5 rows:')
display(weather.tail())

Shape: (781920, 7)

First 5 rows:


,airport_code,date,hour,temperature,precipitation,wind_speed,weather_code
0,PPG,2026-01-01,0,29.4,0.0,10.1,3
1,PPG,2026-01-01,1,29.3,0.1,9.9,51
2,PPG,2026-01-01,2,26.5,0.2,21.1,51
3,PPG,2026-01-01,3,27.6,0.0,10.3,3
4,PPG,2026-01-01,4,28.1,0.0,8.6,3



Last 5 rows:


,airport_code,date,hour,temperature,precipitation,wind_speed,weather_code
781915,EAR,2026-03-31,19,13.2,0.0,23.4,3
781916,EAR,2026-03-31,20,13.6,0.0,22.4,3
781917,EAR,2026-03-31,21,13.6,0.0,21.5,3
781918,EAR,2026-03-31,22,13.4,0.0,20.1,3
781919,EAR,2026-03-31,23,12.4,0.0,18.2,3


In [25]:
print('Required airports:', len(weather_airports))
print('Weather airports:', weather['airport_code'].nunique())

weather_airport_counts = weather['airport_code'].value_counts().sort_index()
print(weather_airport_counts.head())

Required airports: 362
Weather airports: 362
airport_code
ABE    2160
ABI    2160
ABQ    2160
ABR    2160
ABY    2160
Name: count, dtype: int64


In [26]:
wrong_airport_counts = weather_airport_counts[weather_airport_counts != 2160]
print('Airports with unexpected row counts:', len(wrong_airport_counts))
if len(wrong_airport_counts) > 0:
    display(wrong_airport_counts)
else:
    print('All airports have 2,160 hourly records.')

Airports with unexpected row counts: 0
All airports have 2,160 hourly records.


In [27]:
print('Start date:', weather['date'].min())
print('End date:', weather['date'].max())
print('\nMissing values:')
print(weather.isna().sum())

Start date: 2026-01-01
End date: 2026-03-31

Missing values:
airport_code     0
date             0
hour             0
temperature      0
precipitation    0
wind_speed       0
weather_code     0
dtype: int64


In [28]:
duplicate_count = weather.duplicated(subset=['airport_code','date','hour']).sum()
print('Duplicate weather records:', duplicate_count)
print('Negative precipitation:', (weather['precipitation'] < 0).sum())
print('Negative wind speed:', (weather['wind_speed'] < 0).sum())

Duplicate weather records: 0
Negative precipitation: 0
Negative wind speed: 0


In [29]:
print(weather['weather_code'].value_counts().sort_index())

weather_code
0     248825
1      68114
2      54624
3     308276
51     37324
53      9900
55      3120
61      6263
63      4390
65       455
71     24091
73     13291
75      3247
Name: count, dtype: int64


## 20. Check failed airports

In [30]:
failed_df = pd.DataFrame(failed_airports)
print('Failed airports:', len(failed_df))
if len(failed_df) > 0:
    display(failed_df)

Failed airports: 0


## 21. Save ONE combined weather CSV

In [31]:
weather = weather[['airport_code','date','hour','temperature','precipitation','wind_speed','weather_code']]

weather_file = weather_processed_path / 'weather_cleaned.csv'
weather.to_csv(weather_file, index=False)

print('Final weather dataset saved:')
print(weather_file)

Final weather dataset saved:
D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\processed\weather_cleaned.csv


## 22. Final validation report

For 362 airports covering 90 days at 24 observations per day:

**362 × 90 × 24 = 781,920 rows**

The exact row count should be 781,920 if every airport has a complete hourly series.

In [32]:
print('=' * 60)
print('FINAL WEATHER DATA VALIDATION')
print('=' * 60)
print('Total rows:', len(weather))
print('Unique airports:', weather['airport_code'].nunique())
print('Start date:', weather['date'].min())
print('End date:', weather['date'].max())
print('Duplicate records:', weather.duplicated(subset=['airport_code','date','hour']).sum())
print('\nMissing values:')
print(weather.isna().sum())
print('\nNegative precipitation:', (weather['precipitation'] < 0).sum())
print('Negative wind speed:', (weather['wind_speed'] < 0).sum())
print('\nFailed airports:', len(failed_df))
print('=' * 60)

FINAL WEATHER DATA VALIDATION
Total rows: 781920
Unique airports: 362
Start date: 2026-01-01
End date: 2026-03-31
Duplicate records: 0

Missing values:
airport_code     0
date             0
hour             0
temperature      0
precipitation    0
wind_speed       0
weather_code     0
dtype: int64

Negative precipitation: 0
Negative wind speed: 0

Failed airports: 0


# Weather collection completed

The cleaned weather dataset is now stored as **one CSV file**:

`data/processed/weather_cleaned.csv`

Raw API responses remain in:

`data/raw/weather/`

### Do not start weather-vs-delay analysis yet.

The next stage is the careful **flight → airport → weather integration**, including conversion of BTS `CRS_DEP_TIME` and handling the UTC/local-time issue before the merge.